# Tangram IA — entrenar el detector de fichas por forma (versión Kaggle)

| qué | dónde |
|---|---|
| paquete de código | Dataset de Kaggle con `tangram_sintetico.zip` |
| fotos reales para medir | Dataset de Kaggle con `figuras_armadas_unet.zip` |
| dataset sintético generado | `/kaggle/temp/ds` — disco de sesión, **no** se guarda |
| corridas de entrenamiento | `/kaggle/working/Tangram_YOLO_runs/` |
| pesos finales | `/kaggle/working/tangram_formas_v2.pt` |

**Ejecuta las celdas en orden, de arriba abajo, sin saltarte ninguna.** Varias
definen variables que las siguientes usan; si el kernel se reinicia, vuelve a
empezar por la 1.

---

### Qué salió mal en la corrida anterior, y qué cambió aquí

La corrida del 1 de septiembre no entrenó nada aprovechable. Cuatro fallos
encadenados, los cuatro ya cerrados:

1. **El entorno estaba en CPU.** `nvidia-smi` no existía y PyTorch cargó la
   versión `+cpu`. Ahora la primera celda **detiene el notebook** si no hay GPU,
   en vez de dejarlo seguir.
2. **La celda de entrenar falló por dos flags que `entrenar.py` no aceptaba**
   (`--patience`, `--workers`). Ya los acepta, así que el entrenamiento nunca
   llegó a arrancar y eso ya no puede repetirse.
3. **La celda de reanudar entrenó sobre COCO, no sobre el Tangram.** Encontró un
   `last.pt` viejo sin estado de optimizador; Ultralytics avisó por consola,
   empezó un entrenamiento nuevo y —sin `data`— cayó a su dataset por defecto,
   `coco8-seg`: 100 épocas sobre 8 fotos de personas y perros, todas las
   métricas en cero. Ahora se reanuda con `entrenar.py --reanudar`, que
   comprueba el checkpoint antes de tocarlo.
4. **Las rutas de la evaluación estaban mal**: `evaluar_deteccion.py` vive dentro
   del código desempaquetado, no en la raíz; y el zip de test es
   `figuras_armadas_unet.zip`, no `FirgurasArmadas_YOLOv8.zip` —ese otro es un
   dataset distinto, de cajas—. Corregidas, y el modelo viejo ahora viaja dentro
   del propio paquete.

## Lo que cambia respecto a la versión de Colab

No es una traducción de rutas: Kaggle no tiene Drive y su disco funciona de otra
manera. Cuatro diferencias que sí importan.

**1. Los datos entran como *Datasets*, no montando una carpeta.** Hay que subir
los dos zip una vez y añadirlos a este notebook:

- *Create → New Dataset* → sube `vision-service/tangram_sintetico.zip` (37,7 MB,
  el de hoy) → ponle un título, p. ej. `tangram-sintetico`.
- Otro Dataset con `figuras_armadas_unet.zip` (las 114 fotos reales con máscara).
- Aquí en el editor: panel derecho → **+ Add Input** → *Your Datasets* → añade
  los dos.

Aparecen en `/kaggle/input/<slug>/` **en solo lectura**. Kaggle descomprime los
zip al crear el Dataset, así que puede que encuentres los archivos ya sueltos en
vez del `.zip`; la celda 1 acepta las dos formas y no hace falta que sepas cuál
te tocó.

**2. `/kaggle/working` es lo único que se guarda, y solo si guardas versión.**
Tiene 20 GB. Cuando la sesión termina, todo lo demás desaparece —incluido
`/kaggle/temp`—. Por eso las corridas van a `working` y el dataset sintético a
`temp`: 6800 imágenes no tienen por qué viajar en la salida del notebook.

**3. Para conservar los pesos: *Save Version → Quick Save*.** No uses
*Save & Run All (Commit)*: eso vuelve a ejecutar el notebook entero desde cero y
te repite el entrenamiento de tres horas. *Quick Save* fotografía lo que ya hay
en `/kaggle/working`.

**4. Hace falta verificar el teléfono, y bloquea las dos cosas a la vez.** Una
cuenta sin verificar tiene en gris *y* el desplegable del acelerador *y* el
interruptor de internet. El segundo se nota más tarde y peor: `pip install`
falla, y si lo llamaste con `-q` el error se traga y lo que ves dos líneas
después es un `ModuleNotFoundError: No module named 'ultralytics'` que no apunta
a nada. Se verifica en `kaggle.com/settings` → *Phone verification*, y hay que
recargar la página del notebook con Ctrl+F5 para que los menús se enteren.

### Antes de ejecutar nada, en el panel derecho

| ajuste | valor |
|---|---|
| **Accelerator** | `GPU T4 x2` (o `GPU P100`) |
| **Internet** | `On` — sin esto fallan `pip install ultralytics` y la descarga de los pesos base `yolov8s-seg.pt` que hace la sección 5 |
| **Persistence** | `Files only` si quieres que `/kaggle/working` aguante entre sesiones |

Cuota del plan gratuito: unas 30 h de GPU por semana y sesiones de ~9 h. El
entrenamiento son ~2-3 h, así que cabe, pero no de sobra: guarda versión en
cuanto termine.

## Por qué hay que reentrenar

El modelo anterior se entrenó con un dataset defectuoso, y el defecto estaba en
la geometría de referencia del proyecto: en `tangram_validator.PIEZAS_CANONICAS`
el **romboide estaba definido con el mismo polígono que el cuadrado** —cuatro
lados iguales y cuatro ángulos de 90°—. Como `composicion.py` construye las
fichas sintéticas a partir de ahí, **las 6000 imágenes de entrenamiento tenían
dos cuadrados y ningún romboide**: el modelo nunca vio la ficha que ahora se le
pide reconocer.

Se notaba en la medición contra fotos reales: le faltaba el romboide en 58 de
114 fotos y el cuadrado en 56, y le sobraban justo los triángulos pequeños que
los sustituían. No los confundía —nunca había visto uno—.

Ya está corregido y verificado: las siete fichas teselan el cuadrado con
cobertura exacta y solape cero, el cuadrado tiene cuatro ángulos de 90° y el
romboide 45/135. La comprobación del validador ahora revisa **ángulos**, no solo
áreas, que es lo que dejaba pasar el error: cuadrado y romboide miden lo mismo.

**El Dataset de Kaggle tiene que llevar el `tangram_sintetico.zip` nuevo** —el de
37,7 MB, con `models/tangram_piezas_seg_best.pt` dentro—, o vas a regenerar el
mismo dataset defectuoso. La celda 1 lo comprueba y para si le falta algo. La
corrida se llama `tangram_formas_v2` para no pisar la anterior, que queda como
referencia.

## 0. Comprobar que hay GPU

Si esta celda para, ve al panel derecho → **Session options → Accelerator →
GPU T4 x2** y vuelve a ejecutarla. Kaggle reinicia el entorno al cambiarlo, así
que tendrás que empezar de nuevo por aquí.

No sigas sin GPU: el entrenamiento pasa de unas 2-3 horas a varios días, y no se
nota hasta que llevas una hora esperando.

In [ ]:
import subprocess

try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or '(sin salida)')
except FileNotFoundError:
    raise SystemExit(
        'No hay GPU en este entorno.\n'
        'Panel derecho -> Session options -> Accelerator -> GPU T4 x2, y repite esta celda.\n'
        'Si el desplegable esta en gris, te falta verificar el telefono: '
        'perfil -> Settings -> Phone verification.'
    )

In [ ]:
# Ultralytics no viene en la imagen de Kaggle, hay que instalarlo, y para eso el
# interruptor Internet del panel derecho tiene que estar en On.
#
# NO instales torch ni nvidia-*: la imagen ya trae CUDA y cuDNN emparejados con
# el driver, y reinstalar torch te deja la version +cpu.
#
# Aqui no se usa `pip install -q`: con -q, si pip falla, la celda parece que va
# bien y el error sale tres lineas mas abajo como un ModuleNotFoundError que no
# dice nada. Mejor ver el fallo de pip donde ocurre.
import importlib, importlib.util, socket, subprocess, sys


def _hay_internet(host='pypi.org', puerto=443, espera=6):
    try:
        socket.create_connection((host, puerto), espera).close()
        return True
    except OSError:
        return False


if importlib.util.find_spec('ultralytics') is None:
    if not _hay_internet():
        raise SystemExit(
            'Esta sesion no tiene salida a internet, asi que pip no puede bajar\n'
            'ultralytics.\n\n'
            '  Panel derecho -> Session options -> Internet -> On\n\n'
            'Kaggle reinicia la sesion al cambiarlo: cuando vuelva, empieza otra vez\n'
            'por la celda 0.\n\n'
            'Si ese interruptor esta en gris, es lo mismo que te bloquea el acelerador:\n'
            'falta verificar el telefono en kaggle.com/settings -> Phone verification.\n'
            'Una cuenta sin verificar no tiene ni GPU ni internet.'
        )
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics'],
                       text=True, capture_output=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-4000:], file=sys.stderr)
        raise SystemExit(f'pip fallo (codigo {r.returncode}). El motivo esta justo arriba.')
    importlib.invalidate_caches()

try:
    import torch, ultralytics
except ModuleNotFoundError:
    raise SystemExit(
        'ultralytics quedo instalado pero este kernel no lo ve.\n'
        'Run -> Restart session y repite esta celda: ya no reinstalara nada.'
    )

ultralytics.checks()

assert torch.cuda.is_available(), (
    'PyTorch no ve la GPU aunque nvidia-smi responda. Suele ser que se reinstalo '
    'torch en la sesion. Session options -> Factory reset, y repite desde la celda 0.'
)
print('GPU vista por PyTorch:', torch.cuda.get_device_name(0))
print('CUDA', torch.version.cuda, '| cuDNN', torch.backends.cudnn.version(),
      '| GPUs visibles:', torch.cuda.device_count())

## 1. Localizar las entradas y desempaquetar el código

Esta celda sustituye al montaje de Drive de la versión de Colab. Hace cuatro
cosas:

- **Busca los dos zip por nombre en todo `/kaggle/input`**, así que no importa
  qué slug le hayas puesto a cada Dataset. Y si Kaggle los descomprimió al
  subirlos, los reconoce igual por su contenido.
- **Copia el código a `/kaggle/temp`**, disco local de la sesión. `/kaggle/input`
  es de solo lectura y Python quiere escribir sus `__pycache__` ahí.
- **Hace `chdir` al código y deja ahí el directorio de trabajo para todo el
  notebook**: es lo que permite que `python -m sintetico...` y
  `python evaluar_deteccion.py` encuentren sus módulos. La corrida anterior falló
  justo por esto.
- **Comprueba que el paquete sea el nuevo**, listando lo que hay si no lo es.

Descomprime con Python, no con `unzip`: `unzip -q` falla en silencio y deja el
error para dos celdas después.

In [ ]:
import os, shutil, zipfile
from itertools import islice
from pathlib import Path

# ── Donde vive cada cosa en Kaggle ──────────────────────────────────────────
ENTRADA = Path('/kaggle/input')     # solo lectura, lo que anadiste con + Add Input
SALIDA  = Path('/kaggle/working')   # 20 GB, es lo unico que guarda Save Version
SCRATCH = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp/tangram')
SCRATCH.mkdir(parents=True, exist_ok=True)

CODIGO  = SCRATCH / 'tangram_sintetico'      # el codigo, en disco de sesion
DATASET = SCRATCH / 'ds'                     # 6800 imagenes: no queremos que viajen en la salida
RUNS    = SALIDA  / 'Tangram_YOLO_runs'      # las corridas si: aqui sobreviven al Save Version
NOMBRE  = 'tangram_formas_v2'   # v2: primer entrenamiento con el romboide corregido


def _inventario(limite=14):
    '''Lista lo que hay en /kaggle/input, para cuando algo no aparece.'''
    if not ENTRADA.is_dir():
        return '  (no existe /kaggle/input: no has anadido ningun Dataset)'
    lineas = []
    for d in sorted(p for p in ENTRADA.iterdir() if p.is_dir()):
        lineas.append(f'  {d.name}/')
        for x in islice(sorted(d.rglob('*')), limite):
            lineas.append(f'      {x.relative_to(d)}')
        lineas.append('      ...')
    return '\n'.join(lineas) or '  (vacio)'


def localizar(nombre_zip, marca):
    '''Encuentra una entrada venga como .zip o ya descomprimida.

    Kaggle descomprime los archivos al crear el Dataset, asi que segun como se
    haya subido puede estar de las dos formas. `marca` es una ruta relativa que
    tiene que existir dentro para dar por bueno el hallazgo cuando ya viene
    suelta.
    '''
    for z in ENTRADA.glob(f'*/**/{nombre_zip}'):
        return z
    for hit in ENTRADA.glob(f'*/**/{marca}'):
        raiz = hit
        for _ in range(len(Path(marca).parts)):
            raiz = raiz.parent
        return raiz
    return None


ORIGEN_CODIGO = localizar('tangram_sintetico.zip', 'sintetico/generar.py')
ORIGEN_TEST   = localizar('figuras_armadas_unet.zip', 'train/images')

faltan = [n for n, o in (('tangram_sintetico.zip', ORIGEN_CODIGO),
                         ('figuras_armadas_unet.zip', ORIGEN_TEST)) if o is None]
if faltan:
    raise SystemExit(
        'No encuentro en /kaggle/input: ' + ', '.join(faltan) + '\n\n'
        'Subelos como Dataset (Create -> New Dataset) y anadelos a este notebook\n'
        'con el boton "+ Add Input" del panel derecho.\n\n'
        'Ahora mismo /kaggle/input tiene:\n' + _inventario()
    )

print('codigo :', ORIGEN_CODIGO)
print('test   :', ORIGEN_TEST)

# ── Materializar el codigo en disco de sesion ───────────────────────────────
if CODIGO.exists():
    shutil.rmtree(CODIGO)
if ORIGEN_CODIGO.suffix == '.zip':
    with zipfile.ZipFile(ORIGEN_CODIGO) as z:
        z.extractall(SCRATCH)          # el zip ya trae la carpeta tangram_sintetico/
    if not CODIGO.is_dir():            # por si el zip no la trae
        raise SystemExit(f'El zip no creo {CODIGO}. Contenido: '
                         f'{sorted(set(p.split("/")[0] for p in zipfile.ZipFile(ORIGEN_CODIGO).namelist()))}')
else:
    shutil.copytree(ORIGEN_CODIGO, CODIGO)

os.chdir(CODIGO)                 # <- todo el notebook trabaja desde aqui
RUNS.mkdir(parents=True, exist_ok=True)

NECESARIOS = ('sintetico/generar.py', 'sintetico/entrenar.py',
              'evaluar_deteccion.py', 'tangram_validator.py')
faltan = [n for n in NECESARIOS if not Path(n).exists()]
if faltan:
    print('Contenido del paquete (sin fondos_reales):')
    for x in sorted(p for p in CODIGO.rglob('*') if 'fondos_reales' not in str(p)):
        print('   ', x.relative_to(CODIGO))
    raise SystemExit(
        'Al paquete le faltan: ' + ', '.join(faltan) + '\n\n'
        'Es la version antigua. La nueva pesa ~37,7 MB (la vieja, 15) y trae el\n'
        'codigo dentro de sintetico/, mas evaluar_deteccion.py, tangram_validator.py\n'
        'y models/tangram_piezas_seg_best.pt.\n\n'
        'Sube de nuevo el Dataset con:\n'
        '  ...\\tangram-ia\\vision-service\\tangram_sintetico.zip\n\n'
        'Los Datasets de Kaggle se actualizan con "New Version" desde la pagina del\n'
        'Dataset; luego en este notebook, + Add Input -> el Dataset -> elige la\n'
        'version nueva. Y repite esta celda.'
    )

print('\nCodigo en', os.getcwd())
print(sorted(os.listdir('.')))
print('\ncorridas ->', RUNS)
print('dataset  ->', DATASET, '(disco de sesion: NO se guarda con Save Version)')

## 1b. Revisar qué hay ya en la carpeta de corridas

Esta celda existe por el fallo 3. Antes de entrenar mira qué hay en la carpeta
de la corrida y **dice sobre qué datos se entrenó**. Si aparece `coco8-seg` o
clases como `person` / `dog`, esos pesos son los de la corrida fallida: no
sirven y hay que apartarlos antes de seguir.

En Kaggle hay un caso que en Colab no existía: si la sesión anterior terminó,
`/kaggle/working` está vacío y aquí no habrá nada aunque sí hubieras entrenado.
Los pesos de una sesión pasada solo están si guardaste versión; entonces se
recuperan como se explica en la sección 6.

No borra nada por su cuenta.

In [ ]:
import torch
from pathlib import Path

corrida = Path(RUNS) / NOMBRE
ultimo  = corrida / 'weights' / 'last.pt'

if not ultimo.exists():
    print(f'{corrida} esta limpia. Adelante con la seccion 2.')
    previas = list(Path('/kaggle/input').glob('*/**/weights/last.pt'))
    if previas:
        print('\nHay checkpoints en los inputs (de una version guardada):')
        for p in previas:
            print('   ', p)
        print('Si quieres continuar uno de esos, ve a la seccion 6.')
else:
    ck = torch.load(ultimo, map_location='cpu', weights_only=False)
    datos_prev = (ck.get('train_args') or {}).get('data')
    clases     = list(getattr(ck.get('model'), 'names', {}).values())
    epoca      = ck.get('epoch', -1)

    print(f'Hay una corrida en {corrida}')
    print(f'  entrenada sobre : {datos_prev}')
    print(f'  clases          : {clases[:8]}{" ..." if len(clases) > 8 else ""}')
    print(f'  epoca guardada  : {epoca}   (-1 = entrenamiento ya cerrado, NO reanudable)')

    sospechosa = (datos_prev and 'coco' in str(datos_prev).lower()) or \
                 any(c in clases for c in ('person', 'dog', 'horse'))
    if sospechosa:
        print('\n  >>> Estos pesos son de la corrida fallida (COCO, no Tangram).')
        print('  >>> Apartalos con la linea de abajo y vuelve a ejecutar esta celda.')
        print(f"\n      !mv '{corrida}' '{corrida}_FALLIDA_coco8'")
    elif epoca < 0:
        print('\n  Entrenamiento terminado. Para uno nuevo, cambia NOMBRE en la celda 1.')
    else:
        print(f'\n  Reanudable desde la epoca {epoca}: usa la seccion 6, no la 5.')

## 2. Generar el dataset

6000 imágenes de entrenamiento y 800 de validación, a 640 px. Esto se genera en
CPU aunque el entrenamiento vaya en GPU. Kaggle da 4 vCPU (Colab daba 2), así que
son unos **15-25 minutos**. Si quieres probar el circuito completo primero, baja
a `--train 600 --val 100`: sale en 3 minutos y el resto del notebook es idéntico.

Las 7 fichas salen juntas y compartiendo aristas, con color y material sorteados
en cada muestra. Las anotaciones son exactas por construcción: se anota el mismo
polígono que se dibuja.

La semilla queda fija en 1234. **Anótalo:** si la sesión se cae y hay que
regenerar, con la misma semilla sale el mismo dataset y el entrenamiento puede
reanudarse.

La celda se salta la generación si el dataset ya está ahí, que es lo normal
después de reiniciar solo el kernel: `/kaggle/temp` sobrevive al reinicio del
kernel, solo se borra cuando termina la sesión entera.

In [ ]:
from pathlib import Path

if (Path(DATASET) / 'data.yaml').exists():
    print(f'{DATASET} ya existe; no lo regenero.')
    print('Si quieres empezar de cero:  !rm -rf "{DATASET}"')
else:
    get_ipython().system(f'python -m sintetico.generar --salida "{DATASET}"'
                         f' --train 6000 --val 800 --lado 640 --semilla 1234')

## 3. Mirar las muestras

**No te saltes esto.** Un dataset puede estar numéricamente perfecto y ser
visualmente inservible, y eso no lo detecta ninguna métrica.

In [ ]:
from IPython.display import Image, display
display(Image(f'{DATASET}/muestras.jpg'))

## 4. Verificar las anotaciones

Comprueba el formato **y** pasa las anotaciones por el validador geométrico del
proyecto. Si él ve en cada imagen un Tangram completo, sin fichas montadas ni
sueltas, los datos son coherentes con el sistema que va a consumirlos.

Referencia de lo que salió la vez anterior, y que debería repetirse:
solape entre fichas ≈ 0.010 (tolerado 0.05), hueco interior ≈ 0.002.

In [ ]:
!python -m sintetico.verificar "{DATASET}" --muestreo 20

## 5. Entrenar

Los pesos caen en `/kaggle/working/Tangram_YOLO_runs/tangram_formas_v2/`.
Ultralytics escribe `last.pt` en cada época.

Dos cosas propias de Kaggle:

- **`--workers 4`**, porque Kaggle da 4 vCPU. En Colab eran 2. Si ves la RAM
  apretada, bájalo a 2.
- **`--device 0`** usa una sola T4, que es lo probado. Con `GPU T4 x2` puedes
  poner `--device 0,1`; reparte el batch entre las dos tarjetas y va casi al
  doble, pero si algo se comporta raro vuelve a `0` antes de buscar en otra
  parte.

`hsv_h=0.5`, dentro de `entrenar.py`, es lo que impide que el modelo vuelva a
apoyarse en el color: rota el matiz por todo el círculo cromático en cada época.
No lo bajes — es el propósito entero de este entrenamiento.

Si ya existe una corrida con este nombre, `entrenar.py` **para en vez de
sobrescribirla**. Para continuarla, salta a la sección 6.

> **Mientras entrena, no cierres la pestaña sin más.** La sesión de Kaggle sigue
> viva un rato con el navegador cerrado, pero el reloj de las ~9 h no se detiene
> y al agotarse pierdes `/kaggle/working` si no has guardado versión. Cuando
> acabe el entrenamiento, guarda con *Save Version → Quick Save* antes de seguir.

In [ ]:
!python -m sintetico.entrenar \
    --dataset "{DATASET}" \
    --epocas 100 --imgsz 640 --batch 16 --device 0 \
    --salida "{RUNS}" --nombre "{NOMBRE}" \
    --patience 25 --workers 4

## 6. Si la sesión se cortó

Aquí Kaggle y Colab se separan de verdad, así que hay dos casos distintos.

**Caso A — solo se reinició el kernel** (o le diste a *Restart*). El disco de la
sesión sigue ahí: `/kaggle/temp/ds` y `/kaggle/working` intactos. Ejecuta las
celdas 0, 1 y 2 —la 2 verá que el dataset ya existe y no lo regenerará— y luego
la celda de reanudar de abajo. Es cuestión de un minuto.

**Caso B — la sesión terminó** (se agotaron las horas, o cerraste y expiró).
`/kaggle/working` y `/kaggle/temp` se borraron. Los pesos solo existen si
guardaste versión, y entonces se recuperan así:

1. Panel derecho → **+ Add Input** → pestaña *Notebooks* → busca este notebook y
   añade su output.
2. Ejecuta las celdas 0, 1 y 2 (la 2 **sí** regenerará el dataset: mismos 1234,
   mismo dataset).
3. Ejecuta la celda de recuperación de abajo, que copia el `last.pt` del input a
   `RUNS`.
4. Y después la de reanudar.

`--reanudar` no es el `resume=True` pelado: antes de tocar nada comprueba que el
checkpoint lleve estado de optimizador y que se entrenara con este mismo
`data.yaml`. Si algo no cuadra, para y lo dice. Es la guarda que faltaba cuando
el notebook se puso a entrenar sobre COCO.

In [ ]:
# Caso B: traer el checkpoint de una version guardada a /kaggle/working
import shutil
from pathlib import Path

destino = Path(RUNS) / NOMBRE / 'weights'
if (destino / 'last.pt').exists():
    print(f'Ya hay un checkpoint en {destino}. No toco nada.')
else:
    candidatos = sorted(Path('/kaggle/input').glob(f'*/**/{NOMBRE}/weights/last.pt'))
    if not candidatos:
        print('No hay ningun last.pt en los inputs.')
        print('Anade el output de este notebook con + Add Input -> Notebooks,')
        print('o empieza el entrenamiento desde la seccion 5.')
    else:
        origen = candidatos[-1]
        destino.mkdir(parents=True, exist_ok=True)
        for w in origen.parent.glob('*.pt'):
            shutil.copy(w, destino / w.name)
            print(f'copiado {w.name}  ({(destino / w.name).stat().st_size/1048576:.1f} MB)')
        # los csv/png de la corrida ayudan a que las graficas de la seccion 7 salgan completas
        for extra in origen.parent.parent.glob('*.*'):
            shutil.copy(extra, Path(RUNS) / NOMBRE / extra.name)
        print(f'\nCheckpoint listo en {destino}. Ahora la celda de reanudar.')

In [ ]:
!python -m sintetico.entrenar \
    --dataset "{DATASET}" \
    --epocas 100 --imgsz 640 --batch 16 --device 0 \
    --salida "{RUNS}" --nombre "{NOMBRE}" \
    --patience 25 --workers 4 \
    --reanudar

## 7. Resultados del entrenamiento

`results.png` muestra las curvas de pérdida y mAP. Lo que interesa es que
`mAP50-95(M)` —la de máscaras— suba y se estabilice, y que las pérdidas de
validación no se despeguen de las de entrenamiento.

La celda imprime además el mejor mAP alcanzado. **Si sale 0, el entrenamiento no
aprendió nada** y no tiene sentido pasar a la sección 8: revisa antes las
muestras de la sección 3.

In [ ]:
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

base = Path(RUNS) / NOMBRE
for fig in ('results.png', 'confusion_matrix_normalized.png'):
    if (base / fig).exists():
        display(Image(str(base / fig)))

csv = base / 'results.csv'
if csv.exists():
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    col = next((c for c in df.columns if 'mAP50-95(M)' in c), None)
    if col:
        print(f'Mejor mAP50-95 de mascara: {df[col].max():.4f}  (epocas completadas: {len(df)})')
        if df[col].max() == 0:
            print('AVISO: cero. El modelo no aprendio nada; no sigas a la seccion 8.')
else:
    print(f'No hay {csv}. Aun no ha entrenado, o la corrida se llama de otra forma.')

## 8. La medición que de verdad importa

El `val` de arriba es sintético igual que el entrenamiento: dice que el modelo
converge, **no** que funcione sobre una mesa. Esto sí lo dice.

`figuras_armadas_unet.zip` son las 114 fotos reales con la máscara de la silueta.
Se compara la unión de las máscaras detectadas contra la silueta real.

**La línea base a batir**, medida con el modelo que hoy está en producción
(`tangram_piezas_seg_best.pt`, que viaja dentro del propio paquete de código, así
que no hay que buscarlo aparte):

| | `tangram_piezas_seg_best.pt` |
|---|---|
| IoU de silueta (media) | 0.096 |
| IoU ≥ 0.75 | 2 de 114 (2 %) |
| Fichas detectadas | 1.21 de 7 |
| Fotos con las 7 | 2 de 114 (2 %) |

El CSV del detalle por imagen va a `/kaggle/working` para que puedas bajarlo
desde la pestaña *Output* después de guardar versión.

In [ ]:
import os, zipfile
from pathlib import Path

os.chdir(CODIGO)                      # por si el kernel se reinicio

if ORIGEN_TEST.suffix == '.zip':
    RUTA_TEST = Path(SCRATCH) / 'unet_ds'
    if not RUTA_TEST.exists():
        with zipfile.ZipFile(ORIGEN_TEST) as z:
            z.extractall(RUTA_TEST)
else:
    RUTA_TEST = Path(ORIGEN_TEST)     # ya venia descomprimido; se lee tal cual

if not (RUTA_TEST / 'train').is_dir():          # trae una carpeta interna
    hijos = [d for d in RUTA_TEST.iterdir() if d.is_dir() and (d / 'train').is_dir()]
    assert hijos, f'No encuentro train/val/test dentro de {RUTA_TEST}'
    RUTA_TEST = hijos[0]

print('dataset de test:', RUTA_TEST)
for split in ('train', 'val', 'test'):
    d = RUTA_TEST / split / 'images'
    print(f'  {split}: {len(list(d.glob("*"))) if d.is_dir() else 0} imagenes')

In [ ]:
import shlex, sys
from pathlib import Path

nuevos = Path(RUNS) / NOMBRE / 'weights' / 'best.pt'
viejos = Path(CODIGO) / 'models' / 'tangram_piezas_seg_best.pt'
CSV    = Path(SALIDA) / 'comparacion.csv'

assert nuevos.exists(), f'No hay pesos entrenados en {nuevos}. Corre la seccion 5.'

cmd = [sys.executable, 'evaluar_deteccion.py', str(RUTA_TEST),
       '--splits', 'train,val,test',
       '--csv', str(CSV)]
if viejos.exists():
    # el viejo primero y el nuevo en --comparar: asi la tabla sale en ese orden
    cmd += ['--pesos', str(viejos), '--comparar', str(nuevos)]
else:
    print('AVISO: falta el modelo viejo; se mide solo el nuevo, sin comparativa.\n')
    cmd += ['--pesos', str(nuevos)]

orden = ' '.join(shlex.quote(c) for c in cmd)
print(orden, '\n')
# subprocess.run() manda la salida al log del kernel y la celda queda vacia;
# hay que pasar la orden por la shell de IPython para verla aqui.
get_ipython().system(orden)

## 9. Guardar los pesos

En Colab bastaba copiar a Drive. En Kaggle hay que dejar los archivos en
`/kaggle/working` **y guardar versión**, o desaparecen con la sesión.

La celda deja los dos archivos en la raíz de `/kaggle/working` con la convención
de siempre y comprueba que cada uno exista antes de copiarlo. Después:

1. Botón **Save Version** (arriba a la derecha).
2. Elige **Quick Save** — *no* *Save & Run All*, que reentrenaría desde cero.
3. Cuando termine: pestaña **Output** de la versión → descarga
   `tangram_formas_v2.pt`.

La celda avisa además de cuánto ocupa `/kaggle/working`, que tiene 20 GB de
límite. Si te acercas, lo que sobra son las corridas intermedias de Ultralytics;
el dataset sintético no cuenta porque vive en `/kaggle/temp`.

In [ ]:
import shutil
from pathlib import Path

destinos = [
    (Path(RUNS) / NOMBRE / 'weights' / 'best.pt', Path(SALIDA) / f'{NOMBRE}.pt'),
    (Path(SALIDA) / 'comparacion.csv',            Path(SALIDA) / f'{NOMBRE}_comparacion.csv'),
]

for origen, destino in destinos:
    if origen.exists():
        if origen.resolve() != destino.resolve():
            shutil.copy(origen, destino)
        print(f'listo    {destino.name}  ({destino.stat().st_size/1048576:.1f} MB)')
    else:
        print(f'FALTA    {origen}  -> no se copio nada')

ocupado = sum(p.stat().st_size for p in Path(SALIDA).rglob('*') if p.is_file())
print(f'\n/kaggle/working ocupa {ocupado/1073741824:.2f} GB de los 20 GB disponibles.')
print('Ahora: Save Version -> Quick Save. Luego pestana Output para descargar.')

### En tu PC

1. Descarga `tangram_formas_v2.pt` de la pestaña *Output* de la versión guardada
   y ponlo en `vision-service/models/`
2. En `vision-service/.env`: `YOLO_WEIGHTS=models/tangram_formas_v2.pt`
3. Reinicia el servicio y comprueba en `/health` que `yolo_loaded` sea `true`

No hay que tocar el backend ni la app: las clases son las cinco geométricas que
`tangram_validator.resolver_taxonomia()` ya reconoce.

### Y después

Con el modelo nuevo puesto, toca recalibrar el umbral del validador:

```
python tangram_validator.py --calibrar
```

y ajustar `MATCH_IOU` / `CLOSE_IOU` en el `.env` con ese resultado, no con el
0.75 que está hoy.

### Un detalle pendiente en el código

El mensaje de error de `_exigir_gpu()`, dentro de `sintetico/entrenar.py`, sigue
diciendo *«En Colab: Entorno de ejecucion → Cambiar tipo de entorno»*. Funciona
igual, pero si acabas trabajando en Kaggle conviene añadir ahí la ruta de Kaggle
(*Session options → Accelerator*) para no despistar a quien lo lea dentro de seis
meses.